# Tool-Calling Agents

This notebook shows two practical ways to give an AI agent tools:

- Small Python functions for arithmetic.
- A Google Search wrapper for specific or current information.

Each example sets up a model, gives it tools and a clear instruction, asks a question, then checks what the agent did and what it answered.

Before running the examples, configure the required provider API keys in a local `.env` file. Keep secrets out of source control.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# @tool exposes ordinary Python functions to the agent with descriptions and schemas.
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [3]:
# Create the model client that the agent will use to interpret requests and choose tools.
llm = ChatOpenAI(model="gpt-4")

## Example 1: Custom Math Tools

`@tool` turns each Python function into something the agent can call. The function name, type hints, and docstring tell the model what it does and what inputs it needs. The direct `.invoke()` calls below are a quick check that each function works before involving the agent.

In [4]:
# Type hints and the docstring help LangChain build a schema and description
# that the model can use when deciding whether and how to call this tool.
@tool
def add_numbers(a:int,b:int):
    """
        Takes in two numbers and returns the sum
        
        Args:
            a: first number
            b: second number
    """
    return a+b

# Call the tool directly first to check its input schema and result.
add_numbers.invoke({"a":1,"b":2})

@tool
def multiply_numbers(a:int,b:int):
    """
        Takes in two numbers and returns the product
        
        Args:
            a: first number
            b: second number
    """
    return a*b

# These examples show each tool's result independently of the agent.
print(add_numbers.invoke({"a":1,"b":2}))
print(multiply_numbers.invoke({"a":1,"b":2}))

3
2


## Build and Run the Math Agent

The agent combines the model, the two math tools, and an instruction. The user message below asks for a calculation that can use both multiplication and addition.

In [5]:
# create_agent combines a chat model with tools and instructions into an agent.
from langchain.agents import create_agent

In [6]:
# Give the agent both math tools and tell it to use them for calculations.
agent = create_agent(
    model=llm,
    tools=[add_numbers, multiply_numbers],
    system_prompt="You are a helpful math assistant. Use the provided tools for arithmetic and briefly explain the result."
)

### Question

The agent is asked: “What is 23 times 10, plus 5?” The two operations make it easy to see whether it uses the available multiplication and addition tools.

In [7]:
# Send a user message to the agent. The agent can call one or both math tools.
response = agent.invoke({
    "messages":[{"role":"user", "content":"What is 23 times 10, plus 5?"}]
})

### Check the Result

The printed messages show the assistant's tool requests, the values returned by the tools, and the final reply. Check that the returned values lead to the answer. This is the visible tool interaction, not the model's private reasoning.

In [8]:
# Inspect the message trace for assistant tool calls, tool results, and the final response.
for res in response["messages"]:
    print(res)
    print("\n")

content='What is 23 times 10, plus 5?' additional_kwargs={} response_metadata={} id='e98cfdf8-c67a-4d20-8b90-1af3f02f5e94'


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 142, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-EU2xdkaF9G6B0TxhnnmHZpysu5HV6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0f5c8-2578-7bc0-9982-78abb706bcb2-0' tool_calls=[{'name': 'multiply_numbers', 'args': {'a': 23, 'b': 10}, 'id': 'call_PyoyoZcS0WEvOkldECHRlmCn', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata=

In [9]:
# Show only the final message's text after inspecting the full trace above.
print(response["messages"][-1].content)

23 times 10 equals 230. When you add 5 to that, you get a total of 235.


## Example 2: Google Search

`GoogleSerperAPIWrapper` provides the search capability, so we do not have to write the search logic ourselves. The agent gets that callable tool just like the math functions above. This example is useful for specific facts that should be checked against external results. Configure the Groq and Serper API keys in `.env` before running it.

In [10]:
# Use Groq for the language model and Serper to retrieve Google Search results.
from langchain_groq import ChatGroq
from langchain_community.utilities import GoogleSerperAPIWrapper

C:\Users\vishe\AppData\Local\Temp\ipykernel_5420\1179744303.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


In [11]:
# Create the model and search client; the search wrapper reads its API key
# from the environment loaded in the first notebook cell.
search_llm = ChatGroq(model="openai/gpt-oss-20b")
web_search = GoogleSerperAPIWrapper()

In [12]:
# Expose search to the model and ask it to ground answers in results.
search_agent = create_agent(
    model=search_llm,
    tools=[web_search.run],
    system_prompt="Use Google Search for specific or current facts. Base answers on the search results, include a source URL when available, and say when the results do not confirm an answer."
)

### Ask the Search Agent

This asks for a specific address. The agent can search for external information and should include a source URL when the results provide one.

In [13]:
# Ask for a specific fact and request a source when one appears in the results.
search_question = "What is the address of Crown Casino"
search_response = search_agent.invoke({
    "messages": [{"role": "user", "content": search_question}]
})

### Review the Search Evidence

Look for the search tool call, the returned results and URLs, and the final answer. Does the answer match the results? If the results are vague or conflicting, verify the address independently rather than treating the model's wording as proof.

In [14]:
# Print the trace so you can see the search request and returned results.
for search_message in search_response["messages"]:
    print(search_message)
    print("\n")

print("Messages in the agent trace =", len(search_response["messages"]))

content='What is the address of Crown Casino' additional_kwargs={} response_metadata={} id='541d440f-3e0b-4724-9b3d-cec305b28b7d'


content='' additional_kwargs={'reasoning_content': 'We need to answer: "What is the address of Crown Casino". We need to use Google Search to find the address. Then include source URL. Use the function.', 'tool_calls': [{'id': 'fc_367e7e18-7b2c-4bdf-aff3-f5f2dd5e4d61', 'function': {'arguments': '{"query":"Crown Casino address"}', 'name': 'run'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 183, 'total_tokens': 243, 'completion_time': 0.061414144, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.008784505, 'prompt_tokens_details': None, 'queue_time': 0.301008753, 'total_time': 0.070198649}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a

In [15]:
# Display the final assistant answer; earlier messages include intermediate tool activity.
print(search_response["messages"][-1].content)

**Crown Casino (Melbourne, Australia)**  
**Address:** 8 Whiteman Street, Southbank, Victoria 3006, Australia  

Source: Google search results for “Crown Casino Melbourne address” – [https://www.crownmelbourne.com.au](https://www.crownmelbourne.com.au) (displayed address: “8 Whiteman Street, Southbank, VIC 3006”).
